# Введение в MapReduce модель на Python


In [3]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [4]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [5]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [6]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [7]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [8]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [10]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [11]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [12]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [13]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [14]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [15]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [16]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [17]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.2645699731960307)),
 (1, np.float64(2.2645699731960307)),
 (2, np.float64(2.2645699731960307)),
 (3, np.float64(2.2645699731960307)),
 (4, np.float64(2.2645699731960307))]

## Inverted index

In [18]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [19]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [20]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [21]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18), ('it', 18)]),
 (1, [('what', 10)])]

## TeraSort

In [22]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.06471056047151713)),
   (None, np.float64(0.07747388428112001)),
   (None, np.float64(0.10031913062944364)),
   (None, np.float64(0.14013211803964587)),
   (None, np.float64(0.16909800337099956)),
   (None, np.float64(0.17039869447463252)),
   (None, np.float64(0.19458345226354645)),
   (None, np.float64(0.24487910622457754)),
   (None, np.float64(0.31251039786863877)),
   (None, np.float64(0.34567329092447774)),
   (None, np.float64(0.3485886773589778)),
   (None, np.float64(0.38958672987071663)),
   (None, np.float64(0.42309724733670406)),
   (None, np.float64(0.431072176494242)),
   (None, np.float64(0.43255636111482343)),
   (None, np.float64(0.44933954467696713)),
   (None, np.float64(0.45166820967163235))]),
 (1,
  [(None, np.float64(0.5305021705078964)),
   (None, np.float64(0.5957308066555478)),
   (None, np.float64(0.6162065734121057)),
   (None, np.float64(0.6187028120010338)),
   (None, np.float64(0.6483494030215313)),
   (None, np.float64(0.66418

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [30]:
from typing import Iterator
import random

# Генерируем список случайных чисел
input_numbers = [random.randint(1, 1000) for _ in range(20)]
print(f"Входные числа: {input_numbers}")

def RECORDREADER():
    """Читает входные числа и возвращает пары (индекс, число)"""
    for idx, num in enumerate(input_numbers):
        yield (idx, num)

def MAP(key, value):
    """Для каждого числа создаем пару ('max', число)"""
    yield ('max', value)

def REDUCE(key, values: Iterator):
    """Находим максимальное значение среди всех чисел"""
    max_value = float('-inf')
    for v in values:
        if v > max_value:
            max_value = v
    yield (key, max_value)

# Запускаем MapReduce
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)
print(f"Результат: {result}")
print(f"Максимальное число: {result[0][1]}")

Входные числа: [272, 626, 251, 991, 851, 140, 277, 178, 291, 518, 33, 695, 230, 977, 679, 242, 139, 918, 214, 780]
Результат: [('max', 991)]
Максимальное число: 991


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [31]:
from typing import Iterator
import random

# Генерируем список случайных чисел
input_numbers = [random.randint(1, 100) for _ in range(20)]
print(f"Входные числа: {input_numbers}")

def RECORDREADER():
    """Читает входные числа"""
    for idx, num in enumerate(input_numbers):
        yield (idx, num)

def MAP(key, value):
    """
    Для каждого числа создаем пару ('avg', (value, 1)),
    где 1 — это счетчик для одного элемента.
    """
    yield ('avg', (value, 1))

def REDUCE(key, values: Iterator):
    """
    Суммируем все значения и все счетчики отдельно,
    затем делим одно на другое.
    """
    total_sum = 0
    total_count = 0
    for v, count in values:
        total_sum += v
        total_count += count

    if total_count > 0:
        yield (key, total_sum / total_count)
    else:
        yield (key, 0)

# Запускаем MapReduce
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print(f"Результат: {result}")
print(f"Среднее значение: {result[0][1]}")
print(f"Проверка (sum/len): {sum(input_numbers)/len(input_numbers)}")

Входные числа: [12, 60, 43, 13, 11, 19, 47, 77, 54, 93, 58, 32, 5, 9, 63, 29, 36, 94, 100, 79]
Результат: [('avg', 46.7)]
Среднее значение: 46.7
Проверка (sum/len): 46.7


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [32]:
from typing import Iterator

def groupbykey_sort(iterable):
    # Сортируем входную последовательность по ключу k2
    sorted_iterable = sorted(iterable, key=lambda x: x[0])

    if not sorted_iterable:
        return

    current_key = None
    current_values = []

    for k, v in sorted_iterable:
        if k == current_key:
            # Если ключ тот же, добавляем значение в текущую группу
            current_values.append(v)
        else:
            # Если ключ сменился, отдаем накопленную группу кроме первого шага
            if current_key is not None:
                yield (current_key, current_values)
            current_key = k
            current_values = [v]

    # Не забываем отдать последнюю группу
    if current_key is not None:
        yield (current_key, current_values)

# Проверка работы
test_data = [('b', 1), ('a', 2), ('b', 3), ('c', 1), ('a', 5)]

print("Исходные данные:", test_data)
print("Результат группировки через сортировку:")
print(list(groupbykey_sort(test_data)))

Исходные данные: [('b', 1), ('a', 2), ('b', 3), ('c', 1), ('a', 5)]
Результат группировки через сортировку:
[('a', [2, 5]), ('b', [1, 3]), ('c', [1])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [36]:
from typing import Iterator
import random

# Генерируем список чисел с повторениями
input_numbers = [random.randint(1, 10) for _ in range(20)]
print(f"Входные данные (20 чисел): {input_numbers}")

def RECORDREADER():
    """Читает входные числа"""
    for idx, num in enumerate(input_numbers):
        yield (idx, num)

def MAP(key, value):
    """
    Каждый элемент становится ключом.
    """
    yield (value, None)

def REDUCE(element, occurrences: Iterator):
    """
    Группировка уже произошла.
    Мы просто выдаем сам элемент один раз.
    """
    yield element

# Запускаем MapReduce
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print(f"Результат (уникальные элементы): {sorted(result)}")
print(f"Количество уникальных: {len(result)}")

Входные данные (20 чисел): [5, 3, 2, 5, 4, 9, 6, 7, 3, 2, 1, 8, 8, 8, 2, 3, 7, 4, 3, 9]
Результат (уникальные элементы): [1, 2, 3, 4, 5, 6, 7, 8, 9]
Количество уникальных: 9


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [37]:
from typing import Iterator, NamedTuple

# Модель данных
class User(NamedTuple):
    id: int
    name: str
    age: int

input_collection = [
    User(id=1, name='Ivan', age=25),
    User(id=2, name='Maria', age=17),
    User(id=3, name='Anna', age=30),
    User(id=4, name='Petr', age=15)
]

# Условие только совершеннолетние (age >= 18)
def condition(user: User) -> bool:
    return user.age >= 18

def RECORDREADER():
    for user in input_collection:
        yield (user.id, user)

def MAP(_, row: User):
    """
    Map проверяет предикат C.
    Если истина, создается пара (t, t).
    """
    if condition(row):
        yield (row, row)

def REDUCE(key_row, rows: Iterator[User]):
    """
    Функция идентичности.
    Просто возвращает полученный кортеж.
    """
    for row in rows:
        yield row

# Запуск MapReduce
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print("Пользователи, прошедшие фильтр (age >= 18):")
for user in result:
    print(user)

Пользователи, прошедшие фильтр (age >= 18):
User(id=1, name='Ivan', age=25)
User(id=3, name='Anna', age=30)


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [38]:
from typing import Iterator, NamedTuple

# Исходная модель данных
class User(NamedTuple):
    id: int
    name: str
    age: int
    gender: str

input_collection = [
    User(id=1, name='Ivan', age=25, gender='male'),
    User(id=2, name='Maria', age=25, gender='female'),
    User(id=3, name='Anna', age=30, gender='female'),
    User(id=4, name='Petr', age=25, gender='male'), # Дубликат по (age, gender) после проекции
]

# Множество атрибутов S, которые мы оставляем
S = ['age', 'gender']

def RECORDREADER():
    for user in input_collection:
        yield (user.id, user)

def MAP(_, row: User):
    """
    Создаем кортеж t', оставляя только атрибуты из S.
    Возвращаем пару (t', t').
    """
    # Выбираем значения только для атрибутов из списка S
    t_prime = tuple(getattr(row, attr) for attr in S)
    yield (t_prime, t_prime)

def REDUCE(t_prime, values: Iterator):
    """
    Схлопываем дубликаты, из списка идентичных t' возвращаем только один.
    """
    # Нам не важно, сколько раз встретился t_prime, возвращаем его один раз
    yield (t_prime, t_prime)

# Запуск
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print(f"Проекция на атрибуты {S}:")
for key, value in result:
    print(key)

Проекция на атрибуты ['age', 'gender']:
(25, 'male')
(25, 'female')
(30, 'female')


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [39]:
from typing import Iterator, NamedTuple

# Два множества
relation_R = [
    (1, 'Ivan'),
    (2, 'Maria')
]

relation_S = [
    (2, 'Maria'), # Дубликат, есть в R
    (3, 'Anna')
]

# Объединяем их для входа
input_collection = relation_R + relation_S

def RECORDREADER():
    for t in input_collection:
        yield (None, t)

def MAP(_, t):
    """
    Превращаем каждый кортеж t в пару (t, t).
    """
    yield (t, t)

def REDUCE(t, occurrences: Iterator):
    """
    Если кортеж был в обеих таблицах, в occurrences будет [t, t].
    Если в одной — [t].
    В обоих случаях выдаем t только один раз.
    """
    yield (t, t)

# Запуск
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print("Результат Union (R ∪ S):")
for key, value in result:
    print(key)

Результат Union (R ∪ S):
(1, 'Ivan')
(2, 'Maria')
(3, 'Anna')


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [40]:
from typing import Iterator

# Два множества
relation_R = [
    (1, 'Ivan'),
    (2, 'Maria'),
    (4, 'Petr')
]

relation_S = [
    (2, 'Maria'), # Есть в обоих
    (3, 'Anna'),
    (4, 'Petr')   # Есть в обоих
]

# Имитируем чтение из обоих источников
input_collection = relation_R + relation_S

def RECORDREADER():
    for t in input_collection:
        yield (None, t)

def MAP(_, t):
    """
    Превращаем каждый кортеж t в пару (t, t).
    """
    yield (t, t)

def REDUCE(t, occurrences: Iterator):
    """
    Считаем количество вхождений кортежа.
    Если их 2 (значит, кортеж пришел из R и из S), выдаем его.
    """
    count = 0
    for _ in occurrences:
        count += 1

    if count == 2:
        yield (t, t)

# Запуск
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print("Результат Intersection (R ∩ S):")
for key, value in result:
    print(key)

Результат Intersection (R ∩ S):
(2, 'Maria')
(4, 'Petr')


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [41]:
from typing import Iterator

# Два отношения
relation_R = [
    (1, 'Ivan'),
    (2, 'Maria'),
    (4, 'Petr')
]

relation_S = [
    (2, 'Maria'), # Есть в R, нужно исключить
    (3, 'Anna')    # Нет в R, игнорируем
]

def RECORDREADER():
    # Читаем из R с меткой 'R'
    for t in relation_R:
        yield (t, 'R')
    # Читаем из S с меткой 'S'
    for t in relation_S:
        yield (t, 'S')

def MAP(t, relation_name):
    """
    Ключ — сам кортеж t.
    Значение — метка отношения (R или S).
    """
    yield (t, relation_name)

def REDUCE(t, labels: Iterator):
    """
    Если в списке меток есть только 'R',
    значит кортеж уникален для R — выдаем его.
    Если есть 'S', значит кортеж нужно исключить.
    """
    label_list = list(labels)

    # Условие кортеж есть в R и его нет в S
    if label_list == ['R']:
        yield (t, t)

# Запуск
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print("Результат Difference (R - S):")
for key, value in result:
    print(key)

Результат Difference (R - S):
(1, 'Ivan')
(4, 'Petr')


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [42]:
from typing import Iterator, NamedTuple

# Отношение R(A, B)
relation_R = [
    ('Ivan', 10),
    ('Maria', 20),
    ('Anna', 10)
]

# Отношение S(B, C)
relation_S = [
    (10, 'IT'),
    (20, 'HR'),
    (30, 'Legal')
]

def RECORDREADER():
    # Читаем из R
    for a, b in relation_R:
        yield (b, ('R', a))
    # Читаем из S
    for b, c in relation_S:
        yield (b, ('S', c))

def MAP(b, val):
    """
    Ключ — общий атрибут b.
    Значение — кортеж с меткой отношения (R или S) и оставшимся атрибутом.
    """
    yield (b, val)

def REDUCE(b, tagged_values: Iterator):
    """
    Собираем отдельно элементы из R и элементы из S для конкретного b.
    Затем формируем декартово произведение этих списков.
    """
    list_R = []
    list_S = []

    for tag, val in tagged_values:
        if tag == 'R':
            list_R.append(val)
        elif tag == 'S':
            list_S.append(val)

    # Формируем тройки (a, b, c)
    for a in list_R:
        for c in list_S:
            yield (None, (a, b, c))

# Запуск
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print("Результат Natural Join (R ⋈ S):")
for _, triple in result:
    print(triple)

Результат Natural Join (R ⋈ S):
('Ivan', 10, 'IT')
('Anna', 10, 'IT')
('Maria', 20, 'HR')


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [43]:
from typing import Iterator, NamedTuple

# Данные (Отдел, Сумма_сделки, ID_сделки)
input_collection = [
    ('Sales', 100, 1),
    ('IT', 500, 2),
    ('Sales', 200, 3),
    ('IT', 300, 4),
    ('HR', 150, 5)
]

def RECORDREADER():
    for a, b, c in input_collection:
        yield (None, (a, b, c))

def MAP(_, row):
    """
    Группируем по атрибуту 'a' (Отдел).
    Агрегируем атрибут 'b' (Сумма).
    """
    a, b, c = row
    yield (a, b)

def REDUCE(a, b_values: Iterator):
    """
    Ключ 'a' — это группа.
    Применяем агрегацию (SUM).
    """
    total = 0
    for b in b_values:
        total += b
    yield (a, total)

# Запуск
output = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(output)

print("Результат Grouping and Aggregation (SUM по отделам):")
for group, value in result:
    print(f"Группа: {group}, Результат: {value}")

Результат Grouping and Aggregation (SUM по отделам):
Группа: Sales, Результат: 300
Группа: IT, Результат: 800
Группа: HR, Результат: 150


### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [45]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j, k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), w * small_mat[i][j])

def REDUCE(key, values):
  (i, k) = key
  el_value = 0
  for v in values:
    el_value += v
  yield ((i, k), el_value)

Проверьте своё решение

In [46]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [47]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [49]:
import numpy as np
from typing import Iterator

I, J, K = 2, 3, 4

# Генерируем тестовые данные
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

def RECORDREADER():
    """Потоково читает сначала матрицу M, затем матрицу N"""
    for i in range(I):
        for j in range(J):
            yield (('M', i, j), mat_M[i, j])

    for j in range(J):
        for k in range(K):
            yield (('N', j, k), mat_N[j, k])

def MAP(k1, v1):
    """
    Для M(i, j, v) ключом делаем j, значением (M, i, v)
    Для N(j, k, w) ключом делаем j, значением (N, k, w)
    """
    tag = k1[0] # 'M' или 'N'

    if tag == 'M':
        _, i, j = k1
        # Чтобы перемножить m_ij и n_jk, они должны встретиться по ключу j
        yield (j, ('M', i, v1))
    else:
        _, j, k = k1
        yield (j, ('N', k, v1))

def REDUCE(j, values: Iterator):
    """
    На вход приходят все элементы i-тых строк матрицы M
    и k-тых столбцов матрицы N, у которых общий индекс j.
    """
    list_M = []
    list_N = []

    # Распределяем значения по корзинам
    for val in values:
        if val[0] == 'M':
            list_M.append((val[1], val[2]))
        else:
            list_N.append((val[1], val[2]))

    # Генерируем частичные произведения для каждой пары (i, k)
    for i, m_ij in list_M:
        for k, n_jk in list_N:
            yield ((i, k), m_ij * n_jk)

# Результатом первого этапа будет список частичных произведений.
# Чтобы получить финальную матрицу, нужен второй проход MapReduce для суммирования.

def MAP_SUM(k, v):
    yield (k, v)

def REDUCE_SUM(k, values):
    yield (k, sum(values))

# Имитация двухстадийного MapReduce
step1_output = list(MapReduce(RECORDREADER, MAP, REDUCE))

# Создаем Reader для второго этапа на основе выхода первого
def STEP2_READER():
    return step1_output

final_output = list(MapReduce(STEP2_READER, MAP_SUM, REDUCE_SUM))

# Проверка
reference = np.matmul(mat_M, mat_N)

def asmatrix(reduce_output):
    res = np.zeros((I, K))
    for ((i, k), val) in reduce_output:
        res[i, k] = val
    return res

print(f"Результаты совпадают: {np.allclose(reference, asmatrix(final_output))}")

Результаты совпадают: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [51]:
import numpy as np
from typing import Iterator

# Параметры матриц
I, J, K = 2, 3, 5
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

def RECORDREADER_M():
    """Генерирует кортежи для матрицы M: ((row, col), value)"""
    for i in range(I):
        for j in range(J):
            yield (('M', i, j), mat_M[i, j])

def RECORDREADER_N():
    """Генерирует кортежи для матрицы N: ((row, col), value)"""
    for j in range(J):
        for k in range(K):
            yield (('N', j, k), mat_N[j, k])

def COMBINED_READER():
    """Имитирует работу распределенной системы, читающей из разных источников"""
    yield from RECORDREADER_M()
    yield from RECORDREADER_N()

def MAP(coords, value):
    """
    Группируем по общему индексу J.
    Для M: (j, ('M', i, value))
    Для N: (j, ('N', k, value))
    """
    matrix_type = coords[0]
    if matrix_type == 'M':
        _, i, j = coords
        yield (j, ('M', i, value))
    else:
        _, j, k = coords
        yield (j, ('N', k, value))

def REDUCE(j, tagged_values: Iterator):
    """
    Для каждого общего индекса j перемножаем все подходящие пары
    и выдаем промежуточные произведения с ключом (i, k).
    """
    list_M = []
    list_N = []

    for tag, idx, val in tagged_values:
        if tag == 'M':
            list_M.append((idx, val))
        else:
            list_N.append((idx, val))

    # Генерируем частичные произведения
    for i, m_val in list_M:
        for k, n_val in list_N:
            yield ((i, k), m_val * n_val)

# Агрегация сумм
def MAP_AGG(key, value):
    yield (key, value)

def REDUCE_AGG(key, values):
    """Суммируем все произведения для ячейки (i, k)"""
    yield (key, sum(values))

# Запуск MapReduce
# Получаем все частичные произведения
step1_output = MapReduce(COMBINED_READER, MAP, REDUCE)

# Суммируем их по ключу (i, k)
def STEP2_READER():
    return step1_output

final_output = list(MapReduce(STEP2_READER, MAP_AGG, REDUCE_AGG))

# Проверка
def asmatrix(reduce_output):
    mat = np.zeros((I, K))
    for ((i, k), val) in reduce_output:
        mat[i, k] = val
    return mat

reference = np.matmul(mat_M, mat_N)
result_matrix = asmatrix(final_output)

print(f"Матрица M:\n{mat_M.shape}")
print(f"Матрица N:\n{mat_N.shape}")
print(f"\nРезультат совпадает с NumPy: {np.allclose(reference, result_matrix)}")

Матрица M:
(2, 3)
Матрица N:
(3, 5)

Результат совпадает с NumPy: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [55]:
import numpy as np
from typing import Iterator

# Размеры
I, J, K = 3, 4, 5
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

# Имитация множества ридеров
def reader_M_part1():
    for i in range(I // 2):
        for j in range(J):
            yield (('M', i, j), mat_M[i, j])

def reader_M_part2():
    for i in range(I // 2, I):
        for j in range(J):
            yield (('M', i, j), mat_M[i, j])

def reader_N_random_subset():
    all_indices = [(j, k) for j in range(J) for k in range(K)]
    np.random.seed(42)
    np.random.shuffle(all_indices)
    # Первые 70%
    for j, k in all_indices[:int(len(all_indices) * 0.7)]:
        yield (('N', j, k), mat_N[j, k])

def reader_N_remaining():
    all_indices = [(j, k) for j in range(J) for k in range(K)]
    np.random.seed(42)
    np.random.shuffle(all_indices)
    # Оставшиеся 30%
    for j, k in all_indices[int(len(all_indices) * 0.7):]:
        yield (('N', j, k), mat_N[j, k])

def DISTRIBUTED_SUPER_READER():
    """Агрегатор всех ридеров"""
    readers = [reader_M_part1, reader_M_part2, reader_N_random_subset, reader_N_remaining]
    for r in readers:
        yield from r()

# MapReduce
def MAP(coords, value):
    tag, r_or_j, c_or_k = coords
    if tag == 'M':
        yield (c_or_k, ('M', r_or_j, value)) # ключ j, value (M, i, m_ij)
    else:
        yield (r_or_j, ('N', c_or_k, value)) # ключ j, value (N, k, n_jk)

def REDUCE(j, tagged_values):
    list_M = []
    list_N = []
    for val in tagged_values:
        if val[0] == 'M': list_M.append((val[1], val[2]))
        else: list_N.append((val[1], val[2]))

    for i, m_val in list_M:
        for k, n_val in list_N:
            yield ((i, k), m_val * n_val)

def MAP_SUM(k, v): yield (k, v)
def REDUCE_SUM(k, vs): yield (k, sum(vs))

# Запуск и проверка

# Получаем частичные произведения
step1_result = list(MapReduce(DISTRIBUTED_SUPER_READER, MAP, REDUCE))

# Суммируем их по ключу (i, k)
final_res = list(MapReduce(lambda: step1_result, MAP_SUM, REDUCE_SUM))

# Конвертация в матрицу для проверки
def as_matrix(data, rows, cols):
    res = np.zeros((rows, cols))
    for (i, k), v in data:
        res[i, k] = v
    return res

print(f"Результат совпадает: {np.allclose(mat_M @ mat_N, as_matrix(final_res, I, K))}")

Результат совпадает: True


Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

Да, будет. В MapReduce порядок и способ разбиения данных не важны. Стадия Shuffle гарантирует, что все элементы с одинаковым индексом j из всех ридеров соберутся в одном месте для перемножения, а финальная стадия просуммирует все частичные результаты. Главное — чтобы в сумме все ридеры выдали все ненулевые элементы матриц.